# Módulo 35: Introducción a SQL

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m35-introduccion-a-sql.sql](m35-introduccion-a-sql.sql). No es una tarea entregada del programa.

Hasta ahora los datos llegaban en archivos CSV que cargábamos con pandas. En una organización, la mayor parte de la información vive en **bases de datos relacionales**, y el lenguaje para consultarlas es **SQL** (*Structured Query Language*). En esta sección se usa la base de ejemplo AdventureWorks de SQL Server; aquí trabajaremos con una versión mínima en **SQLite**, que viene incluida en Python, para poder ejecutar cada consulta dentro del notebook.

1. Bases de datos relacionales
2. `SELECT` y `FROM`
3. Filtrar filas con `WHERE`
4. `BETWEEN`, `IN` y `LIKE`
5. Ordenar con `ORDER BY`
6. Contar filas: `COUNT(*)` frente a `COUNT(columna)`
7. De T-SQL a SQLite
8. Errores frecuentes
9. Ejercicio

## 1. Bases de datos relacionales

Una base de datos relacional organiza la información en **tablas** que se relacionan entre sí. Cada tabla describe un tipo de entidad (empleados, clientes, órdenes) y tiene un conjunto fijo de columnas con un tipo de dato definido.

| Término | Significado | En AdventureWorks |
| --- | --- | --- |
| Base de datos | Conjunto de tablas y objetos relacionados | `AdventureWorks2022` |
| Esquema | Agrupación lógica de tablas dentro de la base | `HumanResources`, `Sales`, `Person` |
| Tabla | Colección de registros del mismo tipo | `HumanResources.Employee` |
| Fila (registro) | Una entidad concreta | Un empleado |
| Columna (campo) | Un atributo, con un tipo de dato | `HireDate`, de tipo fecha |
| Clave primaria | Columna que identifica cada fila de forma única | `BusinessEntityID` |

SQLite no tiene esquemas como SQL Server: cada base es un solo archivo (o, aquí, una base en memoria). Por eso usaremos nombres planos como `Employee`.

In [1]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")   # base temporal: no se escribe ningún archivo

conn.executescript("""
CREATE TABLE Employee (
    BusinessEntityID  INTEGER PRIMARY KEY,
    LoginID           TEXT,
    JobTitle          TEXT,
    OrganizationLevel INTEGER,      -- NULL para quien no tiene jefe
    Gender            TEXT,
    BirthDate         TEXT,         -- fechas ISO 'AAAA-MM-DD'
    HireDate          TEXT,
    VacationHours     INTEGER,
    SickLeaveHours    INTEGER
);
INSERT INTO Employee VALUES
 (1,  'ken0',     'Chief Executive Officer',           NULL, 'M', '1969-01-29', '2009-01-14', 99, 69),
 (2,  'terri0',   'Vice President of Engineering',     1,    'F', '1971-08-01', '2008-01-31',  1, 20),
 (3,  'roberto0', 'Engineering Manager',               2,    'M', '1974-11-12', '2007-11-11',  2, 21),
 (4,  'rob0',     'Senior Tool Designer',              3,    'M', '1974-12-23', '2007-12-05', 48, 80),
 (5,  'gail0',    'Design Engineer',                   3,    'F', '1952-09-27', '2008-01-06',  5, 22),
 (6,  'jossef0',  'Design Engineer',                   3,    'M', '1959-03-11', '2008-01-24',  6, 23),
 (7,  'dylan0',   'Research and Development Manager',  3,    'M', '1987-02-24', '2009-02-08', 61, 50),
 (8,  'diane1',   'Research and Development Engineer', 4,    'F', '1986-06-05', '2008-12-29', 62, 51),
 (9,  'gigi0',    'Research and Development Engineer', 4,    'F', '1979-01-21', '2009-01-16', 63, 51),
 (10, 'michael6', 'Research and Development Manager',  4,    'M', '1984-11-30', '2009-05-03', 16, 64),
 (11, 'ovidiu0',  'Senior Tool Designer',              3,    'M', '1978-01-17', '2010-12-05',  7, 23),
 (12, 'janice0',  'Tool Designer',                     4,    'F', '1989-05-28', '2010-12-23',  9, 24);
""")


def q(sql):
    """Ejecuta una consulta y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn)


# Estructura de la tabla: nombre y tipo de cada columna
q("PRAGMA table_info(Employee)")[["name", "type", "pk"]]

,name,type,pk
0,BusinessEntityID,INTEGER,1
1,LoginID,TEXT,0
2,JobTitle,TEXT,0
3,OrganizationLevel,INTEGER,0
4,Gender,TEXT,0
5,BirthDate,TEXT,0
6,HireDate,TEXT,0
7,VacationHours,INTEGER,0
8,SickLeaveHours,INTEGER,0


## 2. `SELECT` y `FROM`

Toda consulta de lectura tiene al menos dos partes: `SELECT` indica **qué columnas** se quieren y `FROM` indica **de qué tabla**. El asterisco `*` pide todas las columnas, y `AS` renombra una columna en el resultado (un **alias**).

SQL no distingue mayúsculas en las palabras clave; por convención se escriben en mayúsculas para separarlas de los nombres de tablas y columnas.

In [2]:
q("""
SELECT LoginID, JobTitle AS Puesto, HireDate AS FechaContratacion
FROM Employee
LIMIT 5
""")

,LoginID,Puesto,FechaContratacion
0,ken0,Chief Executive Officer,2009-01-14
1,terri0,Vice President of Engineering,2008-01-31
2,roberto0,Engineering Manager,2007-11-11
3,rob0,Senior Tool Designer,2007-12-05
4,gail0,Design Engineer,2008-01-06


`DISTINCT` elimina las filas repetidas del resultado. Es útil para conocer los valores posibles de una columna:

In [3]:
q("SELECT DISTINCT JobTitle FROM Employee ORDER BY JobTitle")

,JobTitle
0,Chief Executive Officer
1,Design Engineer
2,Engineering Manager
3,Research and Development Engineer
4,Research and Development Manager
5,Senior Tool Designer
6,Tool Designer
7,Vice President of Engineering


## 3. Filtrar filas con `WHERE`

`WHERE` conserva solo las filas que cumplen una condición. Las condiciones se combinan con `AND`, `OR` y `NOT`, y los paréntesis aclaran el orden de evaluación.

| Operador | Significado | Ejemplo |
| --- | --- | --- |
| `=`, `<>` | Igual, distinto (`!=` también se acepta) | `Gender = 'F'` |
| `<`, `<=`, `>`, `>=` | Comparaciones | `VacationHours >= 50` |
| `AND`, `OR`, `NOT` | Combinar condiciones | `Gender = 'M' AND VacationHours > 40` |
| `IS NULL`, `IS NOT NULL` | Valor ausente | `OrganizationLevel IS NULL` |

Los textos y las fechas van entre **comillas simples**.

In [4]:
q("""
SELECT LoginID, Gender, VacationHours
FROM Employee
WHERE Gender = 'F' AND VacationHours > 10
""")

,LoginID,Gender,VacationHours
0,diane1,F,62
1,gigi0,F,63


## 4. `BETWEEN`, `IN` y `LIKE`

Tres atajos muy usados en `WHERE`:

- `x BETWEEN a AND b` equivale a `x >= a AND x <= b`: **incluye ambos extremos**. Con fechas en formato ISO (`'AAAA-MM-DD'`), el orden alfabético coincide con el cronológico, así que funciona igual en SQLite.
- `x IN (v1, v2, ...)` equivale a varias igualdades unidas con `OR`.
- `x LIKE patrón` compara texto con comodines: `%` representa cualquier secuencia de caracteres (incluso vacía) y `_` exactamente un carácter.

In [5]:
# Empleados nacidos entre 1980 y 2000 (ambos años incluidos)
q("""
SELECT LoginID, BirthDate
FROM Employee
WHERE BirthDate BETWEEN '1980-01-01' AND '2000-12-31'
""")

,LoginID,BirthDate
0,dylan0,1987-02-24
1,diane1,1986-06-05
2,michael6,1984-11-30
3,janice0,1989-05-28


In [6]:
# Puestos de una lista y puestos que contienen la palabra 'Manager'
display(q("SELECT LoginID, JobTitle FROM Employee WHERE JobTitle IN ('Design Engineer', 'Tool Designer')"))
q("SELECT LoginID, JobTitle FROM Employee WHERE JobTitle LIKE '%Manager%'")

,LoginID,JobTitle
0,gail0,Design Engineer
1,jossef0,Design Engineer
2,janice0,Tool Designer


,LoginID,JobTitle
0,roberto0,Engineering Manager
1,dylan0,Research and Development Manager
2,michael6,Research and Development Manager


## 5. Ordenar con `ORDER BY`

Sin `ORDER BY`, el orden de las filas **no está garantizado**. `ASC` (por defecto) ordena de menor a mayor y `DESC` de mayor a menor. Con varias columnas, la segunda solo desempata cuando la primera es igual.

Para quedarse con las primeras filas, SQLite usa `LIMIT n` al final; SQL Server usa `TOP n` justo después de `SELECT`.

In [7]:
# Más horas de baja por enfermedad primero; a igualdad, por LoginID
q("""
SELECT LoginID, SickLeaveHours
FROM Employee
ORDER BY SickLeaveHours DESC, LoginID ASC
LIMIT 5
""")

,LoginID,SickLeaveHours
0,rob0,80
1,ken0,69
2,michael6,64
3,diane1,51
4,gigi0,51


La misma consulta en T-SQL:

```sql
SELECT TOP 5 LoginID, SickLeaveHours
FROM HumanResources.Employee
ORDER BY SickLeaveHours DESC, LoginID ASC;
```

## 6. Contar filas: `COUNT(*)` frente a `COUNT(columna)`

`COUNT` es una **función de agregado**: resume muchas filas en un solo valor. Su comportamiento depende del argumento:

| Expresión | Cuenta |
| --- | --- |
| `COUNT(*)` | Todas las filas, tengan o no valores nulos |
| `COUNT(columna)` | Filas en las que `columna` **no es NULL** |
| `COUNT(DISTINCT columna)` | Valores distintos no nulos de `columna` |

`NULL` significa "valor desconocido o ausente". No es cero ni texto vacío, y las funciones de agregado lo ignoran.

In [8]:
q("""
SELECT COUNT(*)                          AS filas,
       COUNT(OrganizationLevel)          AS con_nivel,
       COUNT(DISTINCT OrganizationLevel) AS niveles_distintos,
       COUNT(DISTINCT JobTitle)          AS puestos_distintos
FROM Employee
""")

,filas,con_nivel,niveles_distintos,puestos_distintos
0,12,11,4,8


Hay 12 filas, pero `COUNT(OrganizationLevel)` devuelve 11 porque el director general tiene ese campo en `NULL`. `COUNT` también se combina con `WHERE`: primero se filtra y después se cuenta.

In [9]:
q("""
SELECT COUNT(*) AS hombres
FROM Employee
WHERE Gender = 'M'
""")

,hombres
0,7


## 7. De T-SQL a SQLite

El estándar SQL es común, pero cada motor tiene su dialecto. SQL Server usa **T-SQL** (Transact-SQL). Esta tabla reúne las diferencias que aparecen en esta sección del programa:

| Tarea | T-SQL (SQL Server) | SQLite |
| --- | --- | --- |
| Primeras *n* filas | `SELECT TOP n ...` | `... LIMIT n` |
| Elegir base y separar lotes | `USE AdventureWorks2022;` y `GO` | No existen: se conecta a un archivo |
| Nombres completos | `[AdventureWorks2022].[Sales].[Customer]` | `Customer` |
| Texto largo | `NVARCHAR(MAX)` | `TEXT` |
| Renombrar tabla o columna | `EXEC sp_rename ...` | `ALTER TABLE ... RENAME TO ...` / `RENAME COLUMN ... TO ...` |
| Cambiar tipo o restricción | `ALTER COLUMN`, `DROP/ADD CONSTRAINT` | No existen: se reconstruye la tabla |
| Vaciar una tabla | `TRUNCATE TABLE t` | `DELETE FROM t` |
| Número aleatorio | `RAND()` (entre 0 y 1) | `random()` (entero de 64 bits) |
| Concatenar texto | `a + b` | `a \|\| b` |
| Sustituir NULL | `ISNULL(x, v)` o `COALESCE` | `IFNULL(x, v)` o `COALESCE` |
| Tabla temporal | `#tabla` (local), `##tabla` (global) | `CREATE TEMP TABLE tabla` |
| Año de una fecha | `YEAR(d)` | `strftime('%Y', d)` |
| Subcadena y longitud | `SUBSTRING(s, i, n)`, `LEN(s)` | `substr(s, i, n)`, `length(s)` |
| Fecha y hora actual | `GETDATE()` | `datetime('now')` |

In [10]:
# Algunas equivalencias en acción (versión SQLite)
q("""
SELECT LoginID || ' - ' || JobTitle          AS concatenado,
       strftime('%Y', HireDate)              AS anio_contratacion,
       substr(LoginID, 1, 3)                 AS prefijo,
       length(JobTitle)                      AS largo_puesto,
       IFNULL(OrganizationLevel, 'sin jefe') AS nivel
FROM Employee
LIMIT 3
""")

,concatenado,anio_contratacion,prefijo,largo_puesto,nivel
0,ken0 - Chief Executive Officer,2009,ken,23,sin jefe
1,terri0 - Vice President of Engineering,2008,ter,29,1
2,roberto0 - Engineering Manager,2007,rob,19,2


En T-SQL, la misma consulta sería:

```sql
SELECT TOP 3
       LoginID + ' - ' + JobTitle                         AS concatenado,
       YEAR(HireDate)                                     AS anio_contratacion,
       SUBSTRING(LoginID, 1, 3)                           AS prefijo,
       LEN(JobTitle)                                      AS largo_puesto,
       ISNULL(CAST(OrganizationLevel AS VARCHAR), 'sin jefe') AS nivel
FROM HumanResources.Employee;
```

En T-SQL, `ISNULL` necesita el `CAST` porque el resultado toma el tipo de la columna (un entero) y `'sin jefe'` no se puede convertir a número. SQLite, con su tipado dinámico, no lo exige.

## 8. Errores frecuentes

**Comparar con `= NULL`.** Cualquier comparación con `NULL` da "desconocido", nunca verdadero, así que la consulta no devuelve filas. Se usa `IS NULL`.

In [11]:
display(q("SELECT COUNT(*) AS con_igual FROM Employee WHERE OrganizationLevel = NULL"))
q("SELECT COUNT(*) AS con_is FROM Employee WHERE OrganizationLevel IS NULL")

,con_igual
0,0


,con_is
0,1


**`BETWEEN` con fechas que llevan hora.** Si la columna guarda fecha y hora, `'2010-12-23 09:30:00'` es mayor que `'2010-12-23'` y queda fuera de `BETWEEN ... AND '2010-12-23'`. Con horas, el límite superior se escribe como "menor que el día siguiente".

In [12]:
q("""
SELECT '2010-12-23 09:30:00' BETWEEN '2010-12-01' AND '2010-12-23' AS con_between,
       '2010-12-23 09:30:00' >= '2010-12-01'
   AND '2010-12-23 09:30:00' <  '2010-12-24'                       AS con_menor_que
""")

,con_between,con_menor_que
0,0,1


**Usar comillas dobles para texto.** En SQL estándar, las comillas dobles delimitan **nombres** de columnas, no textos. SQLite, si no encuentra la columna, las acepta como texto; pero si el nombre coincide con una columna, compara columna con columna sin avisar. SQL Server, en cambio, lanza un error.

In [13]:
# "Gender" se interpreta como la columna Gender: la condición es siempre verdadera
display(q('SELECT COUNT(*) AS comillas_dobles FROM Employee WHERE Gender = "Gender"'))
q("SELECT COUNT(*) AS comillas_simples FROM Employee WHERE Gender = 'Gender'")

,comillas_dobles
0,12


,comillas_simples
0,0


**Confiar en el orden sin `ORDER BY`.** El motor puede devolver las filas en cualquier orden, y ese orden puede cambiar entre ejecuciones o versiones. Si el orden importa, se escribe `ORDER BY`.

**Contar con `COUNT(columna)` cuando se querían todas las filas.** Si la columna tiene nulos, el conteo es menor. Para contar registros se usa `COUNT(*)`.

## 9. Ejercicio

Se tiene una tabla de productos:

```python
productos = [
    (680, "HL Road Frame - Black, 58", "Black", 1431.50, "2008-04-30"),
    (706, "HL Road Frame - Red, 58",   "Red",   1431.50, "2008-04-30"),
    (707, "Sport-100 Helmet, Red",     "Red",     34.99, "2011-05-31"),
    (708, "Sport-100 Helmet, Black",   "Black",   34.99, "2011-05-31"),
    (709, "Mountain Bike Socks, M",    "White",    9.50, "2011-05-31"),
    (712, "AWC Logo Cap",              "Multi",    8.99, "2011-05-31"),
    (714, "Long-Sleeve Logo Jersey, M","Multi",   49.99, "2011-05-31"),
    (722, "LL Road Frame - Black, 58", "Black",  337.22, "2011-05-31"),
    (842, "Touring-Panniers, Large",   None,     125.00, "2012-05-30"),
    (879, "All-Purpose Bike Stand",    None,     159.00, "2013-05-30"),
]
```

1. Crea la tabla `Product(ProductID, Name, Color, ListPrice, SellStartDate)` en una base en memoria e inserta las filas.
2. Cuenta los productos y cuántos tienen color registrado.
3. Lista nombre y precio de los productos que empezaron a venderse en 2011, del más caro al más barato.
4. Muestra los productos cuyo nombre contiene `Frame` y cuyo color es `Black` o `Red`.

### Solución

In [14]:
productos = [
    (680, "HL Road Frame - Black, 58", "Black", 1431.50, "2008-04-30"),
    (706, "HL Road Frame - Red, 58",   "Red",   1431.50, "2008-04-30"),
    (707, "Sport-100 Helmet, Red",     "Red",     34.99, "2011-05-31"),
    (708, "Sport-100 Helmet, Black",   "Black",   34.99, "2011-05-31"),
    (709, "Mountain Bike Socks, M",    "White",    9.50, "2011-05-31"),
    (712, "AWC Logo Cap",              "Multi",    8.99, "2011-05-31"),
    (714, "Long-Sleeve Logo Jersey, M","Multi",   49.99, "2011-05-31"),
    (722, "LL Road Frame - Black, 58", "Black",  337.22, "2011-05-31"),
    (842, "Touring-Panniers, Large",   None,     125.00, "2012-05-30"),
    (879, "All-Purpose Bike Stand",    None,     159.00, "2013-05-30"),
]

conn.execute("""
CREATE TABLE Product (
    ProductID INTEGER PRIMARY KEY, Name TEXT, Color TEXT,
    ListPrice REAL, SellStartDate TEXT
)""")
conn.executemany("INSERT INTO Product VALUES (?, ?, ?, ?, ?)", productos)

q("SELECT COUNT(*) AS productos, COUNT(Color) AS con_color FROM Product")

,productos,con_color
0,10,8


In [15]:
q("""
SELECT Name, ListPrice
FROM Product
WHERE SellStartDate BETWEEN '2011-01-01' AND '2011-12-31'
ORDER BY ListPrice DESC
""")

,Name,ListPrice
0,"LL Road Frame - Black, 58",337.22
1,"Long-Sleeve Logo Jersey, M",49.99
2,"Sport-100 Helmet, Red",34.99
3,"Sport-100 Helmet, Black",34.99
4,"Mountain Bike Socks, M",9.50
5,AWC Logo Cap,8.99


In [16]:
q("""
SELECT ProductID, Name, Color
FROM Product
WHERE Name LIKE '%Frame%' AND Color IN ('Black', 'Red')
""")

,ProductID,Name,Color
0,680,"HL Road Frame - Black, 58",Black
1,706,"HL Road Frame - Red, 58",Red
2,722,"LL Road Frame - Black, 58",Black


In [17]:
conn.close()

## Resumen

- Una base relacional organiza los datos en tablas con columnas tipadas; la clave primaria identifica cada fila.
- `SELECT ... FROM ... WHERE ... ORDER BY` es el esqueleto de toda consulta de lectura.
- `BETWEEN` incluye ambos extremos, `IN` resume varias igualdades y `LIKE` busca patrones con `%` y `_`.
- `COUNT(*)` cuenta filas; `COUNT(columna)` ignora los `NULL`, que se filtran con `IS NULL`, nunca con `= NULL`.
- T-SQL y SQLite comparten la base del lenguaje, pero difieren en detalles como `TOP`/`LIMIT`, `+`/`||` o `YEAR`/`strftime`.

En el **Módulo 36** se crean y modifican tablas: tipos de datos, restricciones, `INSERT`, `UPDATE` y `DELETE`.